In [1]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [2]:
import SMI_Methods

In [3]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [4]:
def calculate_scores(model, tokenizer, data, word_list):
    scores = [0, 0, 0, 0]
    for i in range(4):
        sc = SMI_Methods.score_model_bert(model, tokenizer, data[i], word_list[i])
        for j in range(4):
            scores[j] += sc[j]
    return [s / 4 for s in scores]

def run_tests(model, tokenizer):
    langs = ["Polish", "Czech", "Slovak", "Croatian", "Slovene"]
    results = []

    for lang in tqdm(langs):
        word_list = SMI_Methods.prep_words(lang)
        data = SMI_Methods.prep_data(lang, word_list)

        scores_1_line = calculate_scores(model, tokenizer, data[0], word_list)
        scores_3_line = calculate_scores(model, tokenizer, data[1], word_list)
        scores_3u_line = calculate_scores(model, tokenizer, data[2], word_list)

        result_entry = {
            'Language': lang,
            '1 Line - Top 1 Score': scores_1_line[0],
            '1 Line - Top 3 Score': scores_1_line[1],
            '1 Line - FC Combinatorially Optimized Score': scores_1_line[2],
            '1 Line - FC Greedy Algorithm Score': scores_1_line[3],
            '3 Line - Top 1 Score': scores_3_line[0],
            '3 Line - Top 3 Score': scores_3_line[1],
            '3 Line - FC Combinatorially Optimized Score': scores_3_line[2],
            '3 Line - FC Greedy Algorithm Score': scores_3_line[3],
            '3 Line (no fill) - Top 1 Score': scores_3u_line[0],
            '3 Line (no fill) - Top 3 Score': scores_3u_line[1],
            '3 Line (no fill) - FC Combinatorially Optimized Score': scores_3u_line[2],
            '3 Line (no fill) - FC Greedy Algorithm Score': scores_3u_line[3]
        }

        results.append(result_entry)

    df = pd.DataFrame(results)
    return df


In [ ]:
#Polish
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
df = run_tests(model, tokenizer)
df.to_csv("polbert.csv")
del tokenizer
torch.cuda.empty_cache()

In [6]:
del tokenizer
torch.cuda.empty_cache()

In [ ]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
df = run_tests(model, tokenizer)
df.to_csv("Czert.csv")
del tokenizer
torch.cuda.empty_cache()

In [ ]:
#Slovak
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
df = run_tests(model, tokenizer)
df.to_csv("slovakbert.csv")
del tokenizer
torch.cuda.empty_cache()

In [ ]:
#Slovene
tokenizer = AutoTokenizer.from_pretrained("EMBEDDIA/sloberta")
model = AutoModelForMaskedLM.from_pretrained("EMBEDDIA/sloberta").cuda()
df = run_tests(model, tokenizer)
df.to_csv("sloberta.csv")
del tokenizer
torch.cuda.empty_cache()

In [6]:
tokenizer = AutoTokenizer.from_pretrained("classla/bcms-bertic")
model = AutoModelForMaskedLM.from_pretrained("classla/bcms-bertic").cuda()
df = run_tests(model, tokenizer)
df.to_csv("bertic.csv")
del tokenizer
torch.cuda.empty_cache()

Some weights of the model checkpoint at classla/bcms-bertic were not used when initializing ElectraForMaskedLM: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ElectraForMaskedLM were not initialized from the model checkpoint at classla/bcms-bertic and are newly initialized: ['generator_predictions.LayerNorm.bias', 'generator_predictions.LayerNorm.weight', 'generator_pred

  0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [01:01<00:00,  5.10s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:58<00:00,  4.89s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:52<00:00,  4.41s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:58<00:00,  4.86s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:38<00:00, 18.23s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [04:02<00:00, 20.19s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:42<00:00, 18.55s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:45<00:00, 18.76s/it]

100%|██████████████████████████

In [9]:
tokenizer = AutoTokenizer.from_pretrained("classla/xlm-r-bertic")
model = AutoModelForMaskedLM.from_pretrained("classla/xlm-r-bertic").cuda()
df = run_tests(model, tokenizer)
df.to_csv("xlm-bertic.csv")
del tokenizer
torch.cuda.empty_cache()

  0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [01:31<00:00,  7.61s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [01:32<00:00,  7.71s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [01:31<00:00,  7.64s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [01:31<00:00,  7.66s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [04:59<00:00, 24.94s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [05:11<00:00, 25.95s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [05:42<00:00, 28.58s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [05:32<00:00, 27.67s/it]

100%|██████████████████████████

In [11]:
tokenizer = AutoTokenizer.from_pretrained("Andrija/SRoBERTa-F")
model = AutoModelForMaskedLM.from_pretrained("Andrija/SRoBERTa-F").cuda()
df = run_tests(model, tokenizer)
df.to_csv("SRoBERTa.csv")
del tokenizer
torch.cuda.empty_cache()

  0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:52<00:00,  4.36s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:49<00:00,  4.11s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:44<00:00,  3.69s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:46<00:00,  3.89s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:05<00:00, 15.47s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:08<00:00, 15.69s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:04<00:00, 15.41s/it]

100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [03:07<00:00, 15.63s/it]

100%|██████████████████████████